# Onlinekommentar API - Example Notebook

This notebook demonstrates the `onlinekommentar` Python package for the public Onlinekommentar JSON and OAI-PMH APIs.

The package is independently developed and is not official, endorsed by, associated with, or affiliated with Onlinekommentar.

Most cells call the live public APIs. Keep the default rate limit enabled and avoid rerunning the whole notebook repeatedly in tight loops.

## 1. Setup and Installation

In [1]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import onlinekommentar

print(f"onlinekommentar {onlinekommentar.__version__} ready")

onlinekommentar 0.1.0 ready


In [2]:
from onlinekommentar import OnlinekommentarClient

client = OnlinekommentarClient(rate_limit_delay=0.2)

## List Recent Commentaries

In [3]:
results = client.list_commentaries(language="en", page=1)

print(f"Page {results.current_page} of {results.last_page}; total: {results.total}")
for commentary in results.commentaries[:5]:
    authors = ", ".join(author.name for author in commentary.authors)
    print(f"{commentary.date} | {commentary.title} | {authors}")
    print(commentary.html_link)

Page 1 of 8; total: 372
2026-05-18 | Art. 10 FDTA | Nicolas Grieder, Peter Hongler
https://onlinekommentar.ch/en/kommentare/dbg10
2026-05-18 | Art. 11 FDTA | Nicolas Grieder, Peter Hongler
https://onlinekommentar.ch/en/kommentare/dbg11
2026-05-18 | Art. 174 FDTA | Jara Fawer, Peter Hongler
https://onlinekommentar.ch/en/kommentare/dbg174
2026-05-04 | Art. 713 CO | Roman Aus der Au, Tina Hediger
https://onlinekommentar.ch/en/kommentare/or713
2026-04-24 | Art. 33 AMLA | Emmanuel Goldberg, Dominic Robert
https://onlinekommentar.ch/en/kommentare/gwg33


## Search Commentaries

In [4]:
search_results = client.list_commentaries(language="de", search="Datenschutz")

for commentary in search_results.commentaries[:5]:
    act = commentary.legislative_act.title if commentary.legislative_act else "No act"
    print(f"{commentary.title} ({act})")
    print(commentary.html_link)

Art. 33 GwG (Anti-Money Laundering Act)
https://onlinekommentar.ch/de/kommentare/gwg33
Art. 18 DSG (Federal Act on Data Protection)
https://onlinekommentar.ch/de/kommentare/dsg18
Art. 55 DSG (Federal Act on Data Protection)
https://onlinekommentar.ch/de/kommentare/dsg55
Art. 4 DSG (Federal Act on Data Protection)
https://onlinekommentar.ch/de/kommentare/dsg4
Art. 52 DSG (Federal Act on Data Protection)
https://onlinekommentar.ch/de/kommentare/dsg52


## Fetch One Commentary

In [5]:
if search_results.commentaries:
    commentary_id = search_results.commentaries[0].commentary_id
    commentary = client.get_commentary(commentary_id)
    print(commentary.title)
    print(commentary.pdf_link)
    print(commentary.raw.keys())

Art. 33 GwG
https://onlinekommentar.ch/de/kommentare/gwg33/print
dict_keys(['data'])


## OAI-PMH Metadata

In [6]:
identify_xml = client.identify()
print(identify_xml[:500])

<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="https://onlinekommentar.ch/oai2.xsl"?>
<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
  <responseDate>2026-05-19T04:05:57.417648Z</responseDate>
  <request verb="Identify">https://onlinekommentar.ch/oai?verb=Identify</request>
  <Identify>
    <repositor


In [7]:
records_xml = client.list_records(metadata_prefix="oai_dc")
print(records_xml[:500])

<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="https://onlinekommentar.ch/oai2.xsl"?>
<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
  <responseDate>2026-05-23T13:49:42.723627Z</responseDate>
  <request verb="ListRecords" metadataPrefix="oai_dc">https://onlinekommentar.ch/oai?metadataPrefix=oai_dc&am


In [8]:
client.close()